In [ ]:
# Este notebook es para entrenar un yolo con el dataset crudo y otro con el dataset con efectos climaticos

from pathlib import Path
import random
import shutil
import cv2

from src.loading import load_dataset_subset
from src.visualization import preview_sequence_grid
from src.augmentation import apply_dirty_effect
from src.experiments import *

In [2]:
TRACKING_ROOT = Path.cwd()
DETRAC_ROOT = TRACKING_ROOT / "DETRAC_Upload"
MIXED_ROOT = TRACKING_ROOT / "DETRAC_mixed_50_50"


print("TRACKING_ROOT:", TRACKING_ROOT)
print("DETRAC_ROOT existe:", DETRAC_ROOT.exists())
print("MIXED_ROOT existe:", MIXED_ROOT.exists())

TRACKING_ROOT: /Users/fedegutman/Desktop/VisionTrafficGuard/tracking
DETRAC_ROOT existe: True
MIXED_ROOT existe: True


In [3]:
subset_clean = load_dataset_subset(
    base_dir=DETRAC_ROOT,
    split="train",
    percent=5.0,    # porcentaje de los datos cargados
    shuffle=True,
)

clean_image_paths = subset_clean["image_paths"]
clean_label_paths = subset_clean["label_paths"]

len(clean_image_paths), clean_image_paths[0]

Total imágenes en train: 82085 | Usando: 4104 (5.0%)


(4104,
 PosixPath('/Users/fedegutman/Desktop/VisionTrafficGuard/tracking/DETRAC_Upload/images/train/MVI_39931_img00852.jpg'))

In [4]:
EXPERIMENTS.keys()

experiments_to_run = {
    "raw": EXPERIMENTS["raw"],
    "weather": EXPERIMENTS["weather"],
}

experiments_to_run

{'raw': {'mixed_dir': '/Users/fedegutman/Desktop/VisionTrafficGuard/tracking/DETRAC_mixed_50_50',
  'archive_dir': '/Users/fedegutman/Desktop/VisionTrafficGuard/tracking/archive',
  'yaml_path': '/Users/fedegutman/Desktop/VisionTrafficGuard/tracking/configs/detrac_mixed_raw.yaml',
  'run_name': None},
 'weather': {'mixed_dir': '/Users/fedegutman/Desktop/VisionTrafficGuard/tracking/DETRAC_weather',
  'archive_dir': '/Users/fedegutman/Desktop/VisionTrafficGuard/tracking/archive',
  'yaml_path': '/Users/fedegutman/Desktop/VisionTrafficGuard/tracking/configs/detrac_mixed_weather.yaml',
  'run_name': None}}

In [5]:
prepare_mixed_yamls(
    experiments=experiments_to_run
)


===== Preparando YAML para experimento: raw =====
Directorio MIXED: /Users/fedegutman/Desktop/VisionTrafficGuard/tracking/DETRAC_mixed_50_50
Config YOLO creada:
  YAML:   /Users/fedegutman/Desktop/VisionTrafficGuard/tracking/configs/detrac_mixed_raw.yaml
  train:  /Users/fedegutman/Desktop/VisionTrafficGuard/tracking/configs/detrac_mixed_raw_train.txt (3283 imágenes)
  val:    /Users/fedegutman/Desktop/VisionTrafficGuard/tracking/configs/detrac_mixed_raw_val.txt (821 imágenes)

===== Preparando YAML para experimento: weather =====
Directorio MIXED: /Users/fedegutman/Desktop/VisionTrafficGuard/tracking/DETRAC_weather
Config YOLO creada:
  YAML:   /Users/fedegutman/Desktop/VisionTrafficGuard/tracking/configs/detrac_mixed_weather.yaml
  train:  /Users/fedegutman/Desktop/VisionTrafficGuard/tracking/configs/detrac_mixed_weather_train.txt (8752 imágenes)
  val:    /Users/fedegutman/Desktop/VisionTrafficGuard/tracking/configs/detrac_mixed_weather_val.txt (2189 imágenes)


In [6]:
results = train_all_experiments(
    experiments=experiments_to_run,
    base_weights="yolo11s.pt",
    epochs=5,         # podés subirlo después
    imgsz=640,
    batch=8,
)



===== Entrenando experimento: raw =====
YAML: /Users/fedegutman/Desktop/VisionTrafficGuard/tracking/configs/detrac_mixed_raw.yaml
run_name: mixed_raw
New https://pypi.org/project/ultralytics/8.3.230 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.229 🚀 Python-3.13.5 torch-2.9.1 CPU (Apple M1)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/fedegutman/Desktop/VisionTrafficGuard/tracking/configs/detrac_mixed_raw.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width

KeyboardInterrupt: 

In [ ]:
qualitative_comparison_on_archive(
    experiments=experiments_to_run,
    n_images=6,
    conf=0.25,
)
